# 02: Statistical Tests

**Study design:** observational comparison. Sellers were not randomly assigned to channels, so results show association, not causation.

**Hypotheses (set before running any test):**

| # | Question | Test |
|---|---|---|
| H1 | Does activation rate differ by channel? | Chi-square test of independence, Wilson confidence intervals |
| H2 | Among active sellers, does 90-day revenue differ by channel? | Kruskal-Wallis (rank-based, robust to outliers) |
| H3 | Do review scores, order counts, or late delivery rates differ by channel? | Kruskal-Wallis |
| H4 | How large is the paid vs. organic difference in revenue per acquired seller? | Bootstrap confidence interval |

**Settings:** significance level 0.05. Pairwise comparisons use the Holm correction for multiple testing.

In [0]:
%pip install statsmodels
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy import stats
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.multitest import multipletests

df = spark.table("workspace.marts.fact_seller_channel_performance").toPandas()

num_cols = ["revenue_90d", "orders_90d", "avg_review_score_90d", "late_delivery_rate_90d"]
df[num_cols] = df[num_cols].astype(float)
df["is_active_90d"] = df["is_active_90d"].astype(bool)

channel_order = ["organic_search", "paid_search", "unknown", "social", "direct_traffic", "other"]
active = df[df["is_active_90d"]]

ALPHA = 0.05
rng = np.random.default_rng(42)

print(f"All sellers: {len(df)}, active sellers: {len(active)}")

In [0]:
table = pd.crosstab(df["channel_group"], df["is_active_90d"]).loc[channel_order]
chi2, p, dof, expected = stats.chi2_contingency(table)

n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

print(table)
print(f"\nChi-square = {chi2:.2f}, df = {dof}, p = {p:.4f}")
print(f"Cramér's V = {cramers_v:.3f}")
print(f"Smallest expected count = {expected.min():.1f}")

In [0]:
rows = []
for ch in channel_order:
    group = df[df["channel_group"] == ch]
    k, n = int(group["is_active_90d"].sum()), len(group)
    low, high = proportion_confint(k, n, alpha=ALPHA, method="wilson")
    rows.append({"channel": ch, "sellers": n, "active": k,
                 "activation_rate": k / n, "ci_low": low, "ci_high": high})

activation_ci = pd.DataFrame(rows).round(3)
activation_ci

In [0]:
pairs, diffs, pvals = [], [], []
for a, b in combinations(channel_order, 2):
    ga = df[df["channel_group"] == a]
    gb = df[df["channel_group"] == b]
    counts = [int(ga["is_active_90d"].sum()), int(gb["is_active_90d"].sum())]
    nobs = [len(ga), len(gb)]
    _, p = proportions_ztest(counts, nobs)
    pairs.append(f"{a} vs {b}")
    diffs.append(counts[0] / nobs[0] - counts[1] / nobs[1])
    pvals.append(p)

reject, p_holm, _, _ = multipletests(pvals, alpha=ALPHA, method="holm")

pairwise_activation = pd.DataFrame({
    "pair": pairs, "rate_difference": diffs,
    "p_raw": pvals, "p_holm": p_holm, "significant": reject,
}).sort_values("p_raw").round(4)
pairwise_activation

In [0]:
def kruskal_by_channel(data, col):
    groups = [data.loc[data["channel_group"] == ch, col].dropna() for ch in channel_order]
    h, p = stats.kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    eta_sq = (h - k + 1) / (n - k)
    return {"metric": col, "n": n, "H": round(h, 2), "p_value": round(p, 4), "eta_sq": round(eta_sq, 3)}

metrics = ["revenue_90d", "orders_90d", "avg_review_score_90d", "late_delivery_rate_90d"]
kw_results = pd.DataFrame([kruskal_by_channel(active, m) for m in metrics])
kw_results

In [0]:
def bootstrap_ci(values, stat=np.median, n_boot=5000):
    values = np.asarray(values)
    boots = [stat(rng.choice(values, size=len(values), replace=True)) for _ in range(n_boot)]
    return np.percentile(boots, [2.5, 97.5])

rows = []
for ch in channel_order:
    v = active.loc[active["channel_group"] == ch, "revenue_90d"]
    low, high = bootstrap_ci(v)
    rows.append({"channel": ch, "active_sellers": len(v), "median_revenue": v.median(),
                 "ci_low": low, "ci_high": high})

median_ci = pd.DataFrame(rows).round(2)
median_ci

In [0]:
paid = df.loc[df["channel_group"] == "paid_search", "revenue_90d"].values
organic = df.loc[df["channel_group"] == "organic_search", "revenue_90d"].values

boot_diffs = [
    rng.choice(paid, len(paid)).mean() - rng.choice(organic, len(organic)).mean()
    for _ in range(5000)
]
low, high = np.percentile(boot_diffs, [2.5, 97.5])

print(f"Mean 90-day revenue per acquired seller: paid R${paid.mean():,.0f}, organic R${organic.mean():,.0f}")
print(f"Difference (paid minus organic): R${paid.mean() - organic.mean():,.0f}")
print(f"95% bootstrap CI: R${low:,.0f} to R${high:,.0f}")

## Results summary

| Hypothesis | Result | Interpretation |
|---|---|---|
| H1: Activation differs by channel | Chi-square = 11.53, p = 0.042, Cramér's V = 0.131 | Significant but small effect |
| H1 pairwise | Only paid vs. organic survives Holm correction: +15.9 points, adjusted p = 0.048 | Borderline; tested further in the regression |
| H2: Revenue among active sellers differs | Kruskal-Wallis p = 0.57, effect size about 0 | No difference |
| H3: Orders, reviews, late delivery differ | p = 0.96, 0.72, 0.67 | No difference |
| H4: Paid vs. organic revenue per acquired seller | +R\$86, 95% CI: -R\$306 to +R\$449 | Not distinguishable from zero |

**Conclusion:** Paid search sellers are more likely to start selling in their first 90 days (54.8% vs. 38.9% for organic search). Once active, sellers perform similarly regardless of channel. Because revenue is highly skewed, the higher activation rate does not produce a detectable difference in revenue per acquired seller.

**Caveats:** observational data (no random assignment); the paid vs. organic result is borderline after correction; small channels (social, direct traffic, other) have wide confidence intervals.